In [2]:
##Code in this file has been modified from Boris Murmann's github below, and the open-source textbook from Harald Pretl and colleagues:
# https://github.com/bmurmann/Book-on-gm-ID-design/blob/main/starter_files_open_source_tools/gf180mcuD/techsweep_plots_from_mat.ipynb
# Murmann and his colleague have written a book on gm/id-based design, "Systematic Design of Analog CMOS Circuits" (2017)

#Pretl's Open-Source Book: https://iic-jku.github.io/analog-circuit-design/analog_circuit_design.pdf

# Copyright 2024 Harald Pretl
# Licensed under the Apache License, Version 2.0 (the “License”); you may not use this
# file except in compliance with the License. You may obtain a copy of the License at
# http://www.apache.org/licenses/LICENSE-2.0


import numpy as np
import scipy.constants as sc
import matplotlib.pyplot as plt
from pygmid import Lookup as lk

nfet = lk('nfet_03v3.mat')
pfet = lk('pfet_03v3.mat')
VDS1 = 0
VGS1 = 3.3
VSB1 = 0

In [ ]:
#NAND Gate Theory
#Finding Gds requirments
l_Nmos = 0.4 #um
l_Pmos = 0.28 #um
freq_target = 600e3
c_load = 2e-12

#freq = 1 / 2 / t_f
t_f = 1 / 2 / freq_target
print('Fall time =', t_f)

#What is t_f? t_f = 2.3 * c_load / gds
gds_nmos = 2.3 * c_load / t_f
gds_pmos = 2.3 * c_load / t_f
print('gds_target =', gds_nmos)

gds_ref_nmos = nfet.lookup('GDS', L=l_Nmos, VGS = VGS1, VDS=VDS1, VSB=VSB1)
gds_ref_pmos = pfet.lookup('GDS', L=l_Pmos, VGS = VGS1, VDS=VDS1, VSB=VSB1)

print ('gds_ref_nmos =', gds_ref_nmos)

#scale gds values using matlab simulation width to find width with desired gds
w_ref = 5e-6
w_nmos = gds_nmos * w_ref / gds_ref_nmos
print ('Nmos Width =', w_nmos)
 #w_nmos was way to small, thus use switching frequency equations to determine if near min width fit requirements

#nmos 
w_nmos_min = 0.22e-6
gds_new_n = gds_ref_nmos * w_nmos_min / w_ref
print ('gds_nmos =', gds_new_n)
t_new_f = 2.3 * c_load / gds_new_n
print('New Fall Time =', t_new_f)

#pmos
w_pmos_min = w_nmos_min * 2 #factor of 2 comes from rule of thumb current compensation
print('Minimum pmos width =', w_pmos_min)
gds_new_p = gds_ref_pmos * w_pmos_min / w_ref
print ('gds_pmos =', gds_new_p)
t_new_r = 2.3 * c_load / gds_new_p
print('New Rise Time =', t_new_r)

#verify that requirements are met
f_switch = 1 / (t_new_f + t_new_r)

if f_switch > freq_target:
 print('Success! frequency =', f_switch)
else:
 print('Failure! frequency =', f_switch)




Fall time = 8.333333333333333e-07
gds_target = 5.52e-06
gds_ref_nmos = 0.002521
Nmos Width = 1.094803649345498e-08
gds_nmos = 0.00011092399999999998
New Fall Time = 4.146983520248098e-08
Minimum pmos width = 4.4e-07
gds_pmos = 9.9176e-05
New Rise Time = 4.638218923933209e-08
Success! frequency = 11382776.963350786


In [ ]:
#Thresehold estimate
#CAUTION: Real thereshold would need to account for w/l difference from .mat file
vt_nmos = nfet.lookup('VT', L=l_Nmos, VGS = VGS1, VDS=VDS1, VSB=VSB1)
vt_pmos = pfet.lookup('VT', L=l_Pmos, VGS = VGS1, VDS=VDS1, VSB=VSB1)

#if large voltage difference change lengths

print('vt_nmos =', vt_nmos)
print('vt_pmos =', vt_pmos)

#simulation theresholds were 40mV apart

vt_nmos = 0.6587
vt_pmos = 0.7416


In [ ]:
#Inverter Gate Theory

#Attempt same transistor sizes as inverter, but with different Rons from different topology

#Finding Gds requirments
l_Nmos = 0.4
l_Pmos = 0.28
freq_target = 600e3
c_load = 1e-12 

#re-estalbish gds
gds_ref_nmos = nfet.lookup('GDS', L=l_Nmos, VGS = VGS1, VDS=VDS1, VSB=VSB1)
gds_ref_pmos = pfet.lookup('GDS', L=l_Pmos, VGS = VGS1, VDS=VDS1, VSB=VSB1)


#nmos 
w_nmos_min = 0.55e-6 #selected to create matching Vth <80mV difference
gds_new_n = gds_ref_nmos * w_nmos_min / w_ref
print ('gds_nmos =', gds_new_n)

t_new_f = 2.2 * c_load * 2 / (gds_new_n)  # series resistors
print('New Fall Time =', t_new_f)

#pmos
w_pmos_min = w_nmos_min * 2 # 3 comes from rule of thumb current compensation
print('Minimum pmos width =', w_pmos_min)
gds_new_p = gds_ref_pmos * w_pmos_min / w_ref
print ('gds_pmos =', gds_new_p)
t_new_r = 2.2 * (c_load) / gds_new_p # Same gds for high resistance scenario
print('New Rise Time =', t_new_r)

#verify that requirements are met
f_switch = 1 / (t_new_f + t_new_r)

if f_switch > freq_target:
 print('Success! frequency =', f_switch)
else:
 print('Failure! frequency =', f_switch)

 #We succeeded, but the fall time is quite large increase W to remedy

 #Rise and Fall times turn out very inaccurate more adjustments must be made to code
 #simulated values: t_r = 40ns, t_f = 50 ns



gds_nmos = 0.00027730999999999996
New Fall Time = 1.5866719555731857e-08
Minimum pmos width = 1.1e-06
gds_pmos = 0.00024794
New Rise Time = 8.873114463176577e-09
Success! frequency = 40420643.050220504


In [ ]:
#Inverter propagation delay
c_load = 1e-12

#inverter C values
cdd_inv_nmos = 1.781e16
cdd_inv_pmos = 2.949e-16
cgg_inv_nmos = 3.467e-16
cgg_inv_pmos = 5.272e-16
css_inv_nmos = 1.385e-16
css_inv_pmos = 2.049e-16


#NAND gate C values
cdd_NAND_nmos = 4.461e-16
cdd_NAND_pmos = 5.908e-16
cgg_NAND_nmos = 8.670e-16
cgg_NAND_pmos = 1.054e-15
css_NAND_nmos = 3.539e-16
css_NAND_pmos = 4.154e-16


#inverter delay for inverter driving NAND gate 
Ron_inv = 1/0.000139
c_inv_NAND = css_inv_nmos + cdd_inv_pmos + cgg_NAND_nmos + cgg_NAND_pmos
t_pd_inv_NAND = np.log(2) * c_inv_NAND * Ron_inv
print('t_pd_inv_NAND =', t_pd_inv_NAND)

#inverter delay for inverter driving inverter gate  
c_inv_inv = css_inv_nmos + cdd_inv_pmos + cgg_inv_nmos + cgg_inv_pmos
t_pd_inv_inv = np.log(2) * c_inv_inv * Ron_inv
print('t_pd_inv_inv =', t_pd_inv_inv)


#NAND Gate delay - high capacitive scenario
Ron_pmos = 1 / 0.000192
Ron_nmos = 1 / 0.000277

c_NAND_nmos_sd = cdd_NAND_nmos + css_NAND_nmos #capacitance between series nmos mosfets
c_NAND_load = 2 * cdd_NAND_pmos + css_NAND_nmos + cgg_NAND_nmos + cgg_NAND_pmos + c_load #capacitance at NAND gate output. factor of 2 due to 2 pmos in parallel.
t_pd_NAND_fall = np.log(2) * (c_NAND_nmos_sd * Ron_nmos + 2 * c_NAND_load  * Ron_nmos) #factor of 2 for two identical nmosfets
t_pd_NAND_rise = np.log(2) * (c_NAND_load * Ron_pmos)
print('t_pd_NAND_fall =', t_pd_NAND_fall)
print('t_pd_NAND_rise =', t_pd_NAND_rise)


t_pd_inv_NAND = 1.17406167043909e-11
t_pd_inv_inv = 6.519074166518104e-12
t_pd_NAND_fall = 5.023973304417146e-09
t_pd_NAND_rise = 3.6226200197372433e-09


In [ ]:
#snub RC circuit - nmos
# unused

#Parameters
cycles = 5
time = 10e-9
freq_oss = cycles/time
print('ossiclation frequency =', freq_oss * 1e-6, 'MHz')

#Find parasitic inductance
l_p = np.square(1 / (2 * np.pi * freq_oss)) / (css_inv_nmos + cdd_inv_pmos)
print('parasitic inductance =', l_p, 'h')
fr=1/(2* np.pi * np.sqrt(l_p * (css_inv_nmos + cdd_inv_pmos)))
print(fr)
r_s = np.pi*2*freq_oss*l_p*1e-3
l_h = np.square(30)*4e-16
p_l = np.square(3.3)*0.2e-12*freq_oss
print(p_l)
print(l_h)
print(r_s)

ossiclation frequency = 500.0 MHz
parasitic inductance = 0.00023378214961314672 h
499999999.99999994
0.001089
3.6e-13
734.4482837650917
